# درس ۱۳ - حافظه عامل


## راه‌اندازی

این نوت‌بوک نحوه ساخت یک عامل رزرو سفر با **حافظه پایدار** را با استفاده از **چارچوب عامل مایکروسافت** (MAF) نشان می‌دهد.

شما یاد خواهید گرفت که چگونه انواع مختلف حافظه عامل — حافظه کاری، کوتاه‌مدت و بلندمدت — بر نحوه نگهداری و استفاده عامل از اطلاعات در طول گفتگوها تأثیر می‌گذارند.

**پیش‌نیازها:**
- یک پروژه Azure AI Foundry با یک مدل چت مستقر شده (مثلاً `gpt-4o-mini`).
- وارد شده با Azure CLI — دستور `az login` را در ترمینال خود اجرا کنید.
- `AZURE_AI_PROJECT_ENDPOINT` — نقطه پایانی پروژه Azure AI Foundry شما.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — نام مدل مستقر شده شما.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## انواع حافظه عامل

عوامل هوش مصنوعی می‌توانند از انواع مختلف حافظه استفاده کنند که هر یک هدف متفاوتی دارند:

### حافظه کاری
خود رشته مکالمه — پیام‌های رد و بدل شده در یک جلسه. عامل می‌تواند به پیام‌های قبلی در همان رشته مراجعه کند تا انسجام را حفظ کند. در MAF این از طریق **`agent.create_session()`** ساخته می‌شود که یک `AgentSession` را برمی‌گرداند.

### حافظه کوتاه‌مدت
اطلاعاتی که برای مدت یک کار یا جلسه دوام می‌آورد اما به صورت دائمی ذخیره نمی‌شود. به عنوان مثال، عامل ممکن است در طول یک گفتگوی برنامه‌ریزی چند مرحله‌ای حقایقی را جمع‌آوری کند و از آنها برای تولید یک برنامه نهایی استفاده نماید.

### حافظه بلندمدت
ترجیحات و حقایقی که **در طول جلسات مختلف** باقی می‌مانند. یک کاربر بازگشتی نباید مجبور باشد محدودیت‌های غذایی یا سبک سفر خود را تکرار کند. حافظه بلندمدت معمولاً توسط یک ذخیره بیرونی مانند پایگاه داده، فایل، یا نمایه برداری پشتیبانی می‌شود و از طریق ابزارها به عامل ارائه می‌گردد.


## حافظه کاری با سشن‌ها

ساده‌ترین شکل حافظه، سشن مکالمه است. وقتی همان سشن (که از طریق `agent.create_session()` ساخته شده است) را به تماس‌های متوالی `agent.run()` منتقل می‌کنید، عامل کل تاریخچه آن مکالمه را می‌بیند و می‌تواند جزئیات قبلی را به خاطر بسپارد.

بیایید یک عامل مسافرتی بسازیم و حافظه کاری را نمایش دهیم.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

عامل بودجه را به درستی به خاطر آورد زیرا هر دو پیام در یک جلسه مشترک هستند. این همان **حافظه کاری** است — که فقط تا عمر جلسه وجود دارد.

### با یک رشته جدید چه اتفاقی می‌افتد؟

اگر یک جلسه **جدید** ایجاد کنیم، عامل هیچ خاطره‌ای از گفتگوی قبلی ندارد:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## الگوی حافظه بلندمدت

برای به خاطر سپردن ترجیحات کاربر **در طول جلسات مختلف**، به یک مخزن پایدار نیاز داریم که خارج از رشته مکالمه قرار داشته باشد. عامل از طریق **ابزارها** — توابعی که می‌تواند برای ذخیره و بازیابی اطلاعات فرا بخواند — به این مخزن دسترسی پیدا می‌کند.

در زیر یک مخزن ترجیحات ساده در حافظه پیاده‌سازی می‌کنیم (در تولید، این را با یک پایگاه داده یا شاخص برداری پشتیبانی می‌کنید) و آن را به‌عنوان ابزارهایی که عامل می‌تواند استفاده کند، ارائه می‌دهیم.

### معماری
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### سناریوی ۱ — کاربر برای اولین بار سفر سالگرد را رزرو می‌کند

سارا برای اولین بار مراجعه می‌کند. عامل باید ترجیحات او را از طریق ابزارها ذخیره کند و از آن‌ها برای پیشنهاد هتل‌ها استفاده کند.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### سناریو 2 — سارا هفته‌ها بعد بازمی‌گردد

سارا یک **رشته کاملاً جدید** را آغاز می‌کند (شبیه‌سازی یک جلسه جدید). حافظه کاری خالی است، اما فروشگاه ترجیح بلندمدت هنوز اطلاعات او را دارد. عامل باید آن را بازیابی کرده و برای شخصی‌سازی پیشنهادات استفاده کند.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## خلاصه

در این درس شما با سه نوع حافظه‌ عامل و نحوه‌ی پیاده‌سازی آن‌ها با چارچوب Microsoft Agent آشنا شدید:

| نوع حافظه | مکانیزم MAF | طول عمر |
|---|---|---|
| **کاری** | `agent.create_session()` | یک گفتگو |
| **کوتاه‌مدت** | زمینه انباشته شده در یک رشته | یک وظیفه / جلسه |
| **بلندمدت** | ذخیره خارجی که از طریق توابع `@tool` دسترسی دارد | در طول جلسات |

### نکات کلیدی
1. **`agent.create_session()`** حافظه کاری را فراهم می‌کند — عامل کل تاریخچه گفتگو را در یک جلسه می‌بیند.
2. **جلسات جدید زمینه را از دست می‌دهند** — بدون حافظه بلندمدت عامل نمی‌تواند گفتگوهای گذشته را به یاد بیاورد.
3. **توابع `@tool` پل ارتباطی هستند** — آن‌ها به عامل اجازه می‌دهند اطلاعات را در یک ذخیره پایدار ذخیره و بازیابی کند.
4. **شخصی‌سازی با گذشت زمان بهتر می‌شود** — هر چه ترجیحات بیشتر ذخیره شود، توصیه‌های عامل بهتر خواهند بود.

### کاربردهای دنیای واقعی
- **خدمات مشتری**: به خاطر سپردن تاریخچه و ترجیحات مشتری
- **دستیارهای شخصی**: حفظ زمینه در طول روزها یا هفته‌ها
- **مراقبت‌های بهداشتی**: پیگیری اطلاعات و ترجیحات بیماران
- **تجارت الکترونیک**: خرید شخصی‌سازی شده بر اساس تاریخچه

### گام‌های بعدی
- جایگزینی دیکشنری حافظه درون‌حافظه با پایگاه داده یا ذخیره برداری (مثلاً Azure AI Search)
- افزودن انقضای حافظه برای اطلاعات حساس به زمان
- ساخت سیستم‌های چندعامله با حافظه مشترک
- کاوش در دفترچه Cognee برای حافظه پشتیبانی شده توسط نمودار دانش


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی اشتباهات یا نادرستی‌هایی باشند. سند اصلی به زبان بومی خود باید به عنوان منبع معتبر محسوب شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هیچ گونه سوءتفاهم یا برداشت نادرستی که از استفاده این ترجمه ناشی شود، نمی‌باشیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
